In [1]:
#@title Cell 22.1 - Notebook overview
# This cell defines Project C and explains the assembly-availability audit.
# No genome assemblies or nucleotide sequences are downloaded in Notebook 22.

from IPython.display import display, Markdown

display(Markdown(r"""
# Notebook 22: Audit E. coli Assembly Availability

## Project C objective

Project C extends Model 3B by adding a targeted nucleotide-sequence similarity
kernel for AMR-relevant regions.

The model will compare sequences from:

- acquired antibiotic-resistance genes;
- resistance-associated housekeeping and target genes;
- porin genes;
- efflux systems and their regulators;
- relevant promoter or upstream regions.

The representation will distinguish:

- exact alleles;
- known resistance mutations;
- wild-type target sequences;
- novel sequence variants;
- partial sequences;
- missing or uncertain sequence calls.

## What Project C will not do

Project C will not perform whole-genome alignment or construct a whole-genome
similarity kernel.

Genome assemblies will be used only as sources from which the selected
AMR-relevant nucleotide regions can later be extracted.

## Relationship to Model 3B

Project C will preserve the existing model architecture:

- the 26-antibiotic kernel;
- the existing full-gene, core-determinant, point-target, and burden kernels;
- pathogen-kernel embedding;
- 32 pathogen coordinates;
- 26 antibiotic coordinates;
- 832 pathogen–antibiotic interaction features;
- Ridge regression;
- BioSample-grouped pathogen-out validation.

The targeted sequence kernel will be added as one new pathogen-similarity
component. The model will not be redesigned unnecessarily.

## Purpose of Notebook 22

Before developing the targeted sequence kernel, this notebook determines
whether usable genome assemblies are available for the 9,377 MIC-linked
training BioSamples.

For every BioSample, the audit will determine:

1. whether an assembly accession is available;
2. whether more than one assembly is associated with the BioSample;
3. the assembly level and current status;
4. whether the assembly is suitable for later targeted sequence extraction.

## Important restriction

Notebook 22 performs only an availability audit.

It will not:

- download 9,377 genome assemblies;
- extract nucleotide sequences;
- compare pathogen sequences;
- construct a sequence kernel;
- retrain the MIC model.

## Feasibility decision

- At least 95% usable assembly availability: proceed.
- Between 90% and 95%: inspect missingness before deciding.
- Below 90%: stop this path unless another reliable sequence source fills the
  missing assemblies.

## Required input

`14B_expanded_pathogen_feature_matrix.csv`

Only its 9,377 BioSample accessions are required.

## Expected notebook length

Notebook 22 contains **10 cells**.
"""))

print(
    "Transition: The next cell will import the required packages "
    "and define the audit settings and filenames."
)


# Notebook 22: Audit E. coli Assembly Availability

## Project C objective

Project C extends Model 3B by adding a targeted nucleotide-sequence similarity
kernel for AMR-relevant regions.

The model will compare sequences from:

- acquired antibiotic-resistance genes;
- resistance-associated housekeeping and target genes;
- porin genes;
- efflux systems and their regulators;
- relevant promoter or upstream regions.

The representation will distinguish:

- exact alleles;
- known resistance mutations;
- wild-type target sequences;
- novel sequence variants;
- partial sequences;
- missing or uncertain sequence calls.

## What Project C will not do

Project C will not perform whole-genome alignment or construct a whole-genome
similarity kernel.

Genome assemblies will be used only as sources from which the selected
AMR-relevant nucleotide regions can later be extracted.

## Relationship to Model 3B

Project C will preserve the existing model architecture:

- the 26-antibiotic kernel;
- the existing full-gene, core-determinant, point-target, and burden kernels;
- pathogen-kernel embedding;
- 32 pathogen coordinates;
- 26 antibiotic coordinates;
- 832 pathogen–antibiotic interaction features;
- Ridge regression;
- BioSample-grouped pathogen-out validation.

The targeted sequence kernel will be added as one new pathogen-similarity
component. The model will not be redesigned unnecessarily.

## Purpose of Notebook 22

Before developing the targeted sequence kernel, this notebook determines
whether usable genome assemblies are available for the 9,377 MIC-linked
training BioSamples.

For every BioSample, the audit will determine:

1. whether an assembly accession is available;
2. whether more than one assembly is associated with the BioSample;
3. the assembly level and current status;
4. whether the assembly is suitable for later targeted sequence extraction.

## Important restriction

Notebook 22 performs only an availability audit.

It will not:

- download 9,377 genome assemblies;
- extract nucleotide sequences;
- compare pathogen sequences;
- construct a sequence kernel;
- retrain the MIC model.

## Feasibility decision

- At least 95% usable assembly availability: proceed.
- Between 90% and 95%: inspect missingness before deciding.
- Below 90%: stop this path unless another reliable sequence source fills the
  missing assemblies.

## Required input

`14B_expanded_pathogen_feature_matrix.csv`

Only its 9,377 BioSample accessions are required.

## Expected notebook length

Notebook 22 contains **10 cells**.


Transition: The next cell will import the required packages and define the audit settings and filenames.


In [2]:
#@title Cell 22.2 - Import packages and define audit settings
# This cell imports the required packages and defines the fixed cohort size,
# filenames, directories, and feasibility thresholds.

from pathlib import Path
import hashlib
import json
import re
import shutil
import zipfile

import numpy as np
import pandas as pd


EXPECTED_BIOSAMPLES = 9377

PROCEED_THRESHOLD = 0.95
REVIEW_THRESHOLD = 0.90

PATHOGEN_FEATURE_FILENAME = (
    "14B_expanded_pathogen_feature_matrix.csv"
)

WORK_DIRECTORY = Path("/content/notebook22_work")
RESULTS_DIRECTORY = WORK_DIRECTORY / "results"

WORK_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

RESULTS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

ASSEMBLY_ACCESSION_PATTERN = re.compile(
    r"\bGC[AF]_\d+\.\d+\b",
    flags=re.IGNORECASE,
)

print("Notebook 22 settings initialised.")
print(f"Expected BioSamples: {EXPECTED_BIOSAMPLES:,}")
print(f"Proceed threshold: {100 * PROCEED_THRESHOLD:.1f}%")
print(f"Review threshold : {100 * REVIEW_THRESHOLD:.1f}%")

print(
    "\nTransition: The next cell will mount Google Drive and locate "
    "the 14B pathogen table and fixed NCBI metadata release."
)

Notebook 22 settings initialised.
Expected BioSamples: 9,377
Proceed threshold: 95.0%
Review threshold : 90.0%

Transition: The next cell will mount Google Drive and locate the 14B pathogen table and fixed NCBI metadata release.


In [4]:
#@title Cell 22.3 - Locate the cohort and NCBI metadata files
# This cell mounts Google Drive and locates the two existing reference files.
# It does not download any new data.

from google.colab import drive, files

drive.mount("/content/drive")

MYDRIVE_DIRECTORY = Path("/content/drive/MyDrive")

REFERENCE_DIRECTORIES = [
    MYDRIVE_DIRECTORY
    / "Model3_MIC_Project/reference_files",
    MYDRIVE_DIRECTORY
    / "Model3_Ecoli_Inference/reference_files",
]

for directory in REFERENCE_DIRECTORIES:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

pathogen_feature_candidates = sorted(
    path
    for directory in REFERENCE_DIRECTORIES
    for path in directory.rglob(
        PATHOGEN_FEATURE_FILENAME
    )
    if path.is_file()
)

if len(pathogen_feature_candidates) == 0:
    print(
        f"{PATHOGEN_FEATURE_FILENAME} was not found. "
        "Please upload it."
    )

    uploaded_files = files.upload()

    if PATHOGEN_FEATURE_FILENAME not in uploaded_files:
        raise FileNotFoundError(
            f"Required file was not uploaded: "
            f"{PATHOGEN_FEATURE_FILENAME}"
        )

    temporary_path = (
        Path("/content")
        / PATHOGEN_FEATURE_FILENAME
    )

    PATHOGEN_FEATURE_PATH = (
        REFERENCE_DIRECTORIES[0]
        / PATHOGEN_FEATURE_FILENAME
    )

    shutil.copy2(
        temporary_path,
        PATHOGEN_FEATURE_PATH,
    )

elif len(pathogen_feature_candidates) == 1:
    PATHOGEN_FEATURE_PATH = (
        pathogen_feature_candidates[0]
    )

else:
    candidate_table = pd.DataFrame(
        {
            "path": [
                str(path)
                for path in pathogen_feature_candidates
            ],
            "size_mb": [
                round(
                    path.stat().st_size / (1024 ** 2),
                    2,
                )
                for path in pathogen_feature_candidates
            ],
        }
    )

    display(candidate_table)

    raise ValueError(
        "More than one 14B pathogen feature file was found."
    )

metadata_candidates = sorted(
    path
    for path in MYDRIVE_DIRECTORY.rglob(
        "*.amr.metadata.tsv"
    )
    if path.is_file()
)

if len(metadata_candidates) == 0:
    raise FileNotFoundError(
        "No NCBI file ending in '.amr.metadata.tsv' "
        "was found in MyDrive."
    )

if len(metadata_candidates) > 1:
    candidate_table = pd.DataFrame(
        {
            "path": [
                str(path)
                for path in metadata_candidates
            ],
            "size_mb": [
                round(
                    path.stat().st_size / (1024 ** 2),
                    2,
                )
                for path in metadata_candidates
            ],
        }
    )

    display(candidate_table)

    raise ValueError(
        "More than one NCBI AMR metadata release was found."
    )

NCBI_METADATA_PATH = metadata_candidates[0]

input_summary = pd.DataFrame(
    [
        {
            "input": "14B pathogen cohort",
            "path": str(PATHOGEN_FEATURE_PATH),
            "size_mb": round(
                PATHOGEN_FEATURE_PATH.stat().st_size
                / (1024 ** 2),
                2,
            ),
        },
        {
            "input": "NCBI AMR metadata",
            "path": str(NCBI_METADATA_PATH),
            "size_mb": round(
                NCBI_METADATA_PATH.stat().st_size
                / (1024 ** 2),
                2,
            ),
        },
    ]
)

print("Required reference files located.")
display(input_summary)

print(
    "\nTransition: The next cell will identify the BioSample and "
    "assembly-accession fields and build their mapping."
)

Mounted at /content/drive
Required reference files located.


,input,path,size_mb
0,14B pathogen cohort,/content/drive/MyDrive/Model3_MIC_Project/refe...,35.91
1,NCBI AMR metadata,/content/drive/MyDrive/Model3_Ecoli_Inference/...,504.60



Transition: The next cell will identify the BioSample and assembly-accession fields and build their mapping.


In [5]:
#@title Cell 22.4 - Build the BioSample-to-assembly mapping
# This cell identifies assembly accessions in the NCBI metadata and retrieves
# mapping rows only for the 9,377 training BioSamples.

cohort_identifiers = pd.read_csv(
    PATHOGEN_FEATURE_PATH,
    usecols=["biosample"],
    dtype=str,
)

cohort_identifiers["biosample"] = (
    cohort_identifiers["biosample"]
    .str.strip()
    .str.upper()
)

assert len(cohort_identifiers) == EXPECTED_BIOSAMPLES
assert cohort_identifiers["biosample"].is_unique
assert cohort_identifiers["biosample"].notna().all()

cohort_biosamples = set(
    cohort_identifiers["biosample"]
)

metadata_header = pd.read_csv(
    NCBI_METADATA_PATH,
    sep="\t",
    nrows=0,
)

header_columns = metadata_header.columns.tolist()

biosample_candidates = [
    column
    for column in [
        "biosample_acc",
        "biosample",
        "BioSample",
        "biosample_accession",
    ]
    if column in header_columns
]

if len(biosample_candidates) != 1:
    raise ValueError(
        "Could not identify exactly one BioSample field. "
        f"Candidates found: {biosample_candidates}"
    )

BIOSAMPLE_COLUMN = biosample_candidates[0]

# First inspect likely assembly fields by their column names.
named_assembly_candidates = [
    column
    for column in header_columns
    if (
        "assembly" in column.lower()
        or "asm_acc" in column.lower()
    )
]

sample_columns = list(
    dict.fromkeys(
        [BIOSAMPLE_COLUMN]
        + named_assembly_candidates
    )
)

metadata_sample = pd.read_csv(
    NCBI_METADATA_PATH,
    sep="\t",
    dtype=str,
    usecols=sample_columns,
    nrows=100000,
    keep_default_na=False,
    low_memory=False,
)

assembly_match_counts = {}

for column in named_assembly_candidates:
    match_count = int(
        metadata_sample[column]
        .astype(str)
        .str.contains(
            ASSEMBLY_ACCESSION_PATTERN,
            na=False,
        )
        .sum()
    )

    assembly_match_counts[column] = match_count

valid_assembly_candidates = {
    column: count
    for column, count in assembly_match_counts.items()
    if count > 0
}

if not valid_assembly_candidates:
    raise ValueError(
        "The fixed NCBI metadata contains no named field with "
        "GCA/GCF assembly accessions. A separate NCBI assembly "
        "cross-reference will be required."
    )

highest_match_count = max(
    valid_assembly_candidates.values()
)

best_assembly_candidates = [
    column
    for column, count
    in valid_assembly_candidates.items()
    if count == highest_match_count
]

if len(best_assembly_candidates) != 1:
    raise ValueError(
        "More than one possible assembly-accession field was found: "
        f"{best_assembly_candidates}"
    )

ASSEMBLY_COLUMN = best_assembly_candidates[0]

optional_columns = [
    column
    for column in [
        "scientific_name",
        "target_acc",
        "assembly_level",
        "assembly_status",
        "refseq_category",
    ]
    if column in header_columns
]

selected_metadata_columns = list(
    dict.fromkeys(
        [
            BIOSAMPLE_COLUMN,
            ASSEMBLY_COLUMN,
        ]
        + optional_columns
    )
)

mapping_chunks = []
chunks_processed = 0

for metadata_chunk in pd.read_csv(
    NCBI_METADATA_PATH,
    sep="\t",
    dtype=str,
    usecols=selected_metadata_columns,
    chunksize=250000,
    keep_default_na=False,
    low_memory=False,
):
    chunks_processed += 1

    normalized_biosamples = (
        metadata_chunk[BIOSAMPLE_COLUMN]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    matching_rows = metadata_chunk.loc[
        normalized_biosamples.isin(
            cohort_biosamples
        )
    ].copy()

    if matching_rows.empty:
        continue

    matching_rows["biosample"] = (
        matching_rows[BIOSAMPLE_COLUMN]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    mapping_chunks.append(matching_rows)

if not mapping_chunks:
    raise ValueError(
        "None of the 9,377 BioSamples was found in the "
        "NCBI metadata release."
    )

raw_mapping_rows = pd.concat(
    mapping_chunks,
    ignore_index=True,
)

# Extract every valid GCA/GCF accession from the selected field.
assembly_mapping_rows = []

for _, row in raw_mapping_rows.iterrows():
    accessions = ASSEMBLY_ACCESSION_PATTERN.findall(
        str(row[ASSEMBLY_COLUMN])
    )

    for accession in sorted(set(accessions)):
        mapping_row = {
            "biosample": row["biosample"],
            "assembly_accession": accession.upper(),
        }

        for column in optional_columns:
            mapping_row[column] = row[column]

        assembly_mapping_rows.append(
            mapping_row
        )

assembly_mapping = pd.DataFrame(
    assembly_mapping_rows
)

if assembly_mapping.empty:
    raise ValueError(
        "The cohort metadata rows contained no valid GCA/GCF "
        "assembly accessions."
    )

assembly_mapping = (
    assembly_mapping
    .drop_duplicates(
        subset=[
            "biosample",
            "assembly_accession",
        ]
    )
    .sort_values(
        [
            "biosample",
            "assembly_accession",
        ]
    )
    .reset_index(drop=True)
)

print("BioSample-to-assembly mapping constructed.")
print(f"BioSample field : {BIOSAMPLE_COLUMN}")
print(f"Assembly field  : {ASSEMBLY_COLUMN}")
print(f"Metadata chunks : {chunks_processed}")
print(f"Mapping rows    : {len(assembly_mapping):,}")

display(assembly_mapping.head())

print(
    "\nTransition: The next cell will count available, missing, "
    "unique, and ambiguous assembly mappings."
)

BioSample-to-assembly mapping constructed.
BioSample field : biosample_acc
Assembly field  : asm_acc
Metadata chunks : 3
Mapping rows    : 9,058


,biosample,assembly_accession,scientific_name,target_acc
0,SAMN02138598,GCA_000522345.1,Escherichia coli BWH 34,PDT000020627.1
1,SAMN02138602,GCA_000522325.1,Escherichia coli BWH 40,PDT000020626.1
2,SAMN02138649,GCA_000522145.1,Escherichia coli BIDMC 20A,PDT000020617.1
3,SAMN02138668,GCA_000492275.1,Escherichia coli BIDMC 37,PDT000019185.2
4,SAMN02581257,GCA_000692795.1,Escherichia coli UCI 51,PDT000030567.1



Transition: The next cell will count available, missing, unique, and ambiguous assembly mappings.


In [6]:
#@title Cell 22.5 - Summarise assembly availability
# This cell creates one audit row per BioSample and identifies missing or
# ambiguous BioSample-to-assembly mappings.

assembly_counts = (
    assembly_mapping
    .groupby(
        "biosample",
        as_index=False,
    )
    .agg(
        assembly_count=(
            "assembly_accession",
            "nunique",
        ),
        assembly_accessions=(
            "assembly_accession",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
    )
)

assembly_audit = (
    cohort_identifiers
    .merge(
        assembly_counts,
        on="biosample",
        how="left",
        validate="one_to_one",
    )
)

assembly_audit["assembly_count"] = (
    assembly_audit["assembly_count"]
    .fillna(0)
    .astype(int)
)

assembly_audit["assembly_accessions"] = (
    assembly_audit["assembly_accessions"]
    .fillna("")
)

assembly_audit["assembly_available"] = (
    assembly_audit["assembly_count"] >= 1
)

assembly_audit["ambiguous_mapping"] = (
    assembly_audit["assembly_count"] > 1
)

assembly_audit["single_assembly_mapping"] = (
    assembly_audit["assembly_count"] == 1
)

assembly_audit["mapping_status"] = np.select(
    [
        assembly_audit["assembly_count"] == 0,
        assembly_audit["assembly_count"] == 1,
        assembly_audit["assembly_count"] > 1,
    ],
    [
        "missing",
        "single assembly",
        "multiple assemblies",
    ],
    default="unexpected",
)

available_count = int(
    assembly_audit["assembly_available"].sum()
)

missing_count = int(
    (~assembly_audit["assembly_available"]).sum()
)

single_count = int(
    assembly_audit["single_assembly_mapping"].sum()
)

ambiguous_count = int(
    assembly_audit["ambiguous_mapping"].sum()
)

availability_fraction = (
    available_count / EXPECTED_BIOSAMPLES
)

availability_summary = pd.DataFrame(
    [
        {
            "metric": "Training BioSamples",
            "count": EXPECTED_BIOSAMPLES,
            "percentage": 100.0,
        },
        {
            "metric": "At least one assembly",
            "count": available_count,
            "percentage":
                100 * availability_fraction,
        },
        {
            "metric": "Single assembly",
            "count": single_count,
            "percentage":
                100 * single_count
                / EXPECTED_BIOSAMPLES,
        },
        {
            "metric": "Multiple assemblies",
            "count": ambiguous_count,
            "percentage":
                100 * ambiguous_count
                / EXPECTED_BIOSAMPLES,
        },
        {
            "metric": "No assembly found",
            "count": missing_count,
            "percentage":
                100 * missing_count
                / EXPECTED_BIOSAMPLES,
        },
    ]
)

availability_summary["percentage"] = (
    availability_summary["percentage"]
    .round(4)
)

print("Assembly-availability summary:")
display(availability_summary)

print(
    "\nTransition: The next cell will inspect accession types, "
    "assembly metadata, and ambiguous mappings."
)

Assembly-availability summary:


,metric,count,percentage
0,Training BioSamples,9377,100.0000
1,At least one assembly,9058,96.5981
2,Single assembly,9058,96.5981
3,Multiple assemblies,0,0.0000
4,No assembly found,319,3.4019



Transition: The next cell will inspect accession types, assembly metadata, and ambiguous mappings.


In [7]:
#@title Cell 22.6 - Audit assembly accession and metadata quality
# This cell distinguishes RefSeq and GenBank accessions and inspects any
# available assembly-level or assembly-status information.

assembly_mapping["accession_source"] = np.where(
    assembly_mapping["assembly_accession"]
    .str.startswith("GCF_"),
    "RefSeq",
    "GenBank",
)

accession_source_summary = (
    assembly_mapping
    .groupby(
        "accession_source",
        as_index=False,
    )
    .agg(
        unique_assemblies=(
            "assembly_accession",
            "nunique",
        ),
        biosamples=(
            "biosample",
            "nunique",
        ),
    )
)

print("Assembly accession sources:")
display(accession_source_summary)

if "assembly_level" in assembly_mapping.columns:
    assembly_level_summary = (
        assembly_mapping
        .replace(
            {"assembly_level": {"": "not reported"}}
        )
        .groupby(
            "assembly_level",
            as_index=False,
        )
        .agg(
            unique_assemblies=(
                "assembly_accession",
                "nunique",
            ),
            biosamples=(
                "biosample",
                "nunique",
            ),
        )
        .sort_values(
            "biosamples",
            ascending=False,
        )
    )

    print("\nAssembly levels:")
    display(assembly_level_summary)

else:
    assembly_level_summary = pd.DataFrame(
        [
            {
                "assembly_level":
                    "not available in fixed metadata",
                "unique_assemblies":
                    assembly_mapping[
                        "assembly_accession"
                    ].nunique(),
                "biosamples":
                    assembly_mapping[
                        "biosample"
                    ].nunique(),
            }
        ]
    )

    print(
        "\nAssembly level is not included in the fixed "
        "NCBI metadata release."
    )

if "assembly_status" in assembly_mapping.columns:
    assembly_status_summary = (
        assembly_mapping
        .replace(
            {"assembly_status": {"": "not reported"}}
        )
        .groupby(
            "assembly_status",
            as_index=False,
        )
        .agg(
            unique_assemblies=(
                "assembly_accession",
                "nunique",
            ),
            biosamples=(
                "biosample",
                "nunique",
            ),
        )
    )

    print("\nAssembly statuses:")
    display(assembly_status_summary)

else:
    assembly_status_summary = pd.DataFrame(
        [
            {
                "assembly_status":
                    "not available in fixed metadata",
                "unique_assemblies":
                    assembly_mapping[
                        "assembly_accession"
                    ].nunique(),
                "biosamples":
                    assembly_mapping[
                        "biosample"
                    ].nunique(),
            }
        ]
    )

    print(
        "\nCurrent assembly status is not included in the "
        "fixed metadata and must be checked before sequence extraction."
    )

ambiguous_mappings = assembly_audit.loc[
    assembly_audit["ambiguous_mapping"]
].copy()

missing_mappings = assembly_audit.loc[
    ~assembly_audit["assembly_available"]
].copy()

print(
    f"\nAmbiguous BioSamples: {len(ambiguous_mappings):,}"
)
display(ambiguous_mappings.head(20))

print(
    f"\nMissing BioSamples: {len(missing_mappings):,}"
)
display(missing_mappings.head(20))

print(
    "\nTransition: The next cell will test whether assembly "
    "missingness is associated with the existing AMR feature burden."
)

Assembly accession sources:


,accession_source,unique_assemblies,biosamples
0,GenBank,9058,9058



Assembly level is not included in the fixed NCBI metadata release.

Current assembly status is not included in the fixed metadata and must be checked before sequence extraction.

Ambiguous BioSamples: 0


,biosample,assembly_count,assembly_accessions,assembly_available,ambiguous_mapping,single_assembly_mapping,mapping_status



Missing BioSamples: 319


,biosample,assembly_count,assembly_accessions,assembly_available,ambiguous_mapping,single_assembly_mapping,mapping_status
118,SAMN04014925,0,,False,False,False,missing
158,SAMN04279645,0,,False,False,False,missing
159,SAMN04279646,0,,False,False,False,missing
160,SAMN04279647,0,,False,False,False,missing
161,SAMN04279648,0,,False,False,False,missing
162,SAMN04279650,0,,False,False,False,missing
235,SAMN05170070,0,,False,False,False,missing
236,SAMN05170083,0,,False,False,False,missing
237,SAMN05170086,0,,False,False,False,missing
238,SAMN05170113,0,,False,False,False,missing



Transition: The next cell will test whether assembly missingness is associated with the existing AMR feature burden.


In [8]:
#@title Cell 22.7 - Examine missingness across AMR feature burden
# This cell checks whether BioSamples without assemblies systematically have
# fewer recorded AMR findings than BioSamples with assemblies.

burden_columns = [
    "full_determinant_count",
    "core_determinant_count",
    "core_gene_count",
    "core_point_mutation_count",
    "partial_call_count",
]

available_feature_columns = pd.read_csv(
    PATHOGEN_FEATURE_PATH,
    nrows=0,
).columns.tolist()

missing_burden_columns = [
    column
    for column in burden_columns
    if column not in available_feature_columns
]

if missing_burden_columns:
    raise ValueError(
        "The 14B feature table is missing burden columns: "
        f"{missing_burden_columns}"
    )

burden_table = pd.read_csv(
    PATHOGEN_FEATURE_PATH,
    usecols=[
        "biosample",
        *burden_columns,
    ],
)

burden_table["biosample"] = (
    burden_table["biosample"]
    .astype(str)
    .str.strip()
    .str.upper()
)

missingness_audit = assembly_audit.merge(
    burden_table,
    on="biosample",
    how="left",
    validate="one_to_one",
)

if missingness_audit[
    burden_columns
].isna().any().any():
    raise ValueError(
        "Some burden features could not be aligned."
    )

burden_by_availability = (
    missingness_audit
    .groupby(
        "assembly_available",
        as_index=False,
    )[burden_columns]
    .mean()
)

burden_by_availability.insert(
    1,
    "biosamples",
    missingness_audit
    .groupby("assembly_available")
    .size()
    .to_numpy(),
)

burden_by_availability[burden_columns] = (
    burden_by_availability[burden_columns]
    .round(4)
)

print("Mean AMR burden by assembly availability:")
display(burden_by_availability)

print(
    "\nThis table is descriptive only. It indicates whether "
    "missing assemblies are concentrated among pathogens with "
    "different AMR-record profiles."
)

print(
    "\nTransition: The next cell will apply the predefined "
    "feasibility thresholds and state the Project C decision."
)

Mean AMR burden by assembly availability:


,assembly_available,biosamples,full_determinant_count,core_determinant_count,core_gene_count,core_point_mutation_count,partial_call_count
0,False,319,16.5423,12.6270,8.3260,4.3009,0.3824
1,True,9058,8.0962,5.0902,3.4098,1.6804,0.1201



This table is descriptive only. It indicates whether missing assemblies are concentrated among pathogens with different AMR-record profiles.

Transition: The next cell will apply the predefined feasibility thresholds and state the Project C decision.


In [9]:
#@title Cell 22.8 - Apply the Project C feasibility rule
# This cell applies the predefined assembly-availability thresholds.
# It does not automatically begin sequence extraction.

if availability_fraction >= PROCEED_THRESHOLD:
    feasibility_decision = "proceed"

    feasibility_explanation = (
        "At least 95% of the 9,377 BioSamples have an assembly "
        "accession. Project C can proceed to current-accession "
        "validation and targeted-locus planning."
    )

elif availability_fraction >= REVIEW_THRESHOLD:
    feasibility_decision = "review missingness"

    feasibility_explanation = (
        "Assembly availability is between 90% and 95%. "
        "Missingness and ambiguous mappings must be examined "
        "before Project C proceeds."
    )

else:
    feasibility_decision = "stop"

    feasibility_explanation = (
        "Fewer than 90% of the BioSamples have an assembly "
        "accession. The targeted-sequence path should stop unless "
        "another reliable sequence source fills the gap."
    )

# Ambiguous mappings require resolution even when overall coverage passes.
if (
    feasibility_decision == "proceed"
    and ambiguous_count > 0
):
    operational_note = (
        f"{ambiguous_count:,} BioSamples have multiple assembly "
        "accessions. A deterministic assembly-selection rule is "
        "required before sequence extraction."
    )
else:
    operational_note = (
        "No additional ambiguity warning changes the threshold decision."
    )

feasibility_summary = pd.DataFrame(
    [
        {
            "metric": "Assembly availability",
            "value": round(
                100 * availability_fraction,
                4,
            ),
            "unit": "percent",
        },
        {
            "metric": "Missing BioSamples",
            "value": missing_count,
            "unit": "BioSamples",
        },
        {
            "metric": "Ambiguous BioSamples",
            "value": ambiguous_count,
            "unit": "BioSamples",
        },
        {
            "metric": "Threshold decision",
            "value": feasibility_decision,
            "unit": "",
        },
    ]
)

print("Project C feasibility decision:")
display(feasibility_summary)

print(feasibility_explanation)
print(operational_note)

print(
    "\nTransition: The next cell will save the complete audit "
    "tables and validation summary."
)

Project C feasibility decision:


,metric,value,unit
0,Assembly availability,96.5981,percent
1,Missing BioSamples,319,BioSamples
2,Ambiguous BioSamples,0,BioSamples
3,Threshold decision,proceed,


At least 95% of the 9,377 BioSamples have an assembly accession. Project C can proceed to current-accession validation and targeted-locus planning.
No additional ambiguity warning changes the threshold decision.

Transition: The next cell will save the complete audit tables and validation summary.


In [10]:
#@title Cell 22.9 - Save and validate the assembly audit
# This cell validates the audit totals and writes all Notebook 22 results.

assert len(assembly_audit) == EXPECTED_BIOSAMPLES

assert (
    available_count + missing_count
    == EXPECTED_BIOSAMPLES
)

assert (
    single_count + ambiguous_count + missing_count
    == EXPECTED_BIOSAMPLES
)

assert assembly_audit["biosample"].is_unique

assembly_mapping.to_csv(
    RESULTS_DIRECTORY
    / "22_biosample_assembly_mapping_rows.csv",
    index=False,
)

assembly_audit.to_csv(
    RESULTS_DIRECTORY
    / "22_biosample_assembly_availability_audit.csv",
    index=False,
)

availability_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_assembly_availability_summary.csv",
    index=False,
)

accession_source_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_assembly_accession_source_summary.csv",
    index=False,
)

assembly_level_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_assembly_level_summary.csv",
    index=False,
)

assembly_status_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_assembly_status_summary.csv",
    index=False,
)

ambiguous_mappings.to_csv(
    RESULTS_DIRECTORY
    / "22_ambiguous_biosample_assembly_mappings.csv",
    index=False,
)

missing_mappings.to_csv(
    RESULTS_DIRECTORY
    / "22_missing_biosample_assemblies.csv",
    index=False,
)

burden_by_availability.to_csv(
    RESULTS_DIRECTORY
    / "22_amr_burden_by_assembly_availability.csv",
    index=False,
)

feasibility_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_project_c_feasibility_summary.csv",
    index=False,
)

notebook_configuration = {
    "project": "Project C",
    "analysis": "E. coli assembly availability audit",
    "expected_biosamples": EXPECTED_BIOSAMPLES,
    "proceed_threshold": PROCEED_THRESHOLD,
    "review_threshold": REVIEW_THRESHOLD,
    "assembly_source": str(NCBI_METADATA_PATH),
    "assembly_column": ASSEMBLY_COLUMN,
    "whole_genome_comparison": False,
    "planned_sequence_scope":
        "targeted AMR-relevant nucleotide regions",
    "feasibility_decision": feasibility_decision,
}

with open(
    RESULTS_DIRECTORY
    / "22_audit_configuration.json",
    "w",
    encoding="utf-8",
) as configuration_file:
    json.dump(
        notebook_configuration,
        configuration_file,
        indent=2,
    )

validation_summary = pd.DataFrame(
    [
        {
            "metric": "Training BioSamples",
            "value": EXPECTED_BIOSAMPLES,
        },
        {
            "metric": "BioSamples audited",
            "value": len(assembly_audit),
        },
        {
            "metric": "BioSamples with assembly",
            "value": available_count,
        },
        {
            "metric": "BioSamples without assembly",
            "value": missing_count,
        },
        {
            "metric": "Ambiguous mappings",
            "value": ambiguous_count,
        },
        {
            "metric": "Availability percentage",
            "value": round(
                100 * availability_fraction,
                4,
            ),
        },
        {
            "metric": "Feasibility decision",
            "value": feasibility_decision,
        },
        {
            "metric": "Validation status",
            "value": "passed",
        },
    ]
)

validation_summary.to_csv(
    RESULTS_DIRECTORY
    / "22_validation_summary.csv",
    index=False,
)

print("Notebook 22 validation passed.")
display(validation_summary)

print(
    "\nTransition: The final cell will package and download "
    "the complete assembly-availability audit."
)

Notebook 22 validation passed.


,metric,value
0,Training BioSamples,9377
1,BioSamples audited,9377
2,BioSamples with assembly,9058
3,BioSamples without assembly,319
4,Ambiguous mappings,0
5,Availability percentage,96.5981
6,Feasibility decision,proceed
7,Validation status,passed



Transition: The final cell will package and download the complete assembly-availability audit.


In [11]:
#@title Cell 22.10 - Package and download Notebook 22 outputs
# This final cell packages every saved audit result into one ZIP archive.

from google.colab import files

OUTPUT_ARCHIVE_NAME = (
    "22_ecoli_assembly_availability_audit_outputs.zip"
)

OUTPUT_ARCHIVE_PATH = (
    WORK_DIRECTORY / OUTPUT_ARCHIVE_NAME
)

if OUTPUT_ARCHIVE_PATH.exists():
    OUTPUT_ARCHIVE_PATH.unlink()

result_files = sorted(
    path
    for path in RESULTS_DIRECTORY.iterdir()
    if path.is_file()
)

if not result_files:
    raise FileNotFoundError(
        "No Notebook 22 result files were found."
    )

with zipfile.ZipFile(
    OUTPUT_ARCHIVE_PATH,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
) as output_archive:
    for result_file in result_files:
        output_archive.write(
            result_file,
            arcname=result_file.name,
        )

if not zipfile.is_zipfile(OUTPUT_ARCHIVE_PATH):
    raise ValueError(
        "The Notebook 22 output archive is invalid."
    )

print("Notebook 22 completed.")
print(f"Files packaged: {len(result_files)}")
print(f"ZIP archive   : {OUTPUT_ARCHIVE_PATH.name}")
print(
    f"ZIP size      : "
    f"{OUTPUT_ARCHIVE_PATH.stat().st_size / (1024 ** 2):.2f} MB"
)

print(f"\nProject C decision: {feasibility_decision}")
print(feasibility_explanation)

files.download(
    str(OUTPUT_ARCHIVE_PATH)
)

Notebook 22 completed.
Files packaged: 12
ZIP archive   : 22_ecoli_assembly_availability_audit_outputs.zip
ZIP size      : 0.14 MB

Project C decision: proceed
At least 95% of the 9,377 BioSamples have an assembly accession. Project C can proceed to current-accession validation and targeted-locus planning.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>